# SDAHU: raw windows (1 and sparse 100), sampled train, streaming val/test

Этот ноутбук:
- загружает `train/val/test` из CSV с `MultiIndex(['fault_type', 'Datetime'])`
- корректно извлекает target из `DataFrame` с тем же `MultiIndex`
- строит **raw-window** признаки как все значения в окне, **внутри каждой траектории `fault_type`**
- для **train** генерирует окна только для **стратифицированного сэмпла**
- для **val/test** делает **streaming inference батчами**, без материализации всех окон в память
- сравнивает `Dummy`, `ExtraTrees`, `HistGradientBoosting` для окна `1`, `10`, `100` и `100`, где берутся только позиции `0, 10, ..., 90`.


In [ ]:
import gc
import json
import os
import math
import time
import warnings
from os.path import join

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss, classification_report


In [ ]:

folder = r'/home/ilya_treyvish/projects/lbnl_fdd/data/processed/SDAHU'

train_df = pd.read_csv(join(folder, 'train_df.csv'), index_col=[0, 1], parse_dates=[1])
train_target = pd.read_csv(join(folder, 'train_target.csv'), index_col=[0, 1], parse_dates=[1])

val_df = pd.read_csv(join(folder, 'val_df.csv'), index_col=[0, 1], parse_dates=[1])
val_target = pd.read_csv(join(folder, 'val_target.csv'), index_col=[0, 1], parse_dates=[1])

test_df = pd.read_csv(join(folder, 'test_df.csv'), index_col=[0, 1], parse_dates=[1])
test_target = pd.read_csv(join(folder, 'test_target.csv'), index_col=[0, 1], parse_dates=[1])

print("train_df:", train_df.shape)
print("train_target:", train_target.shape)
print("val_df:", val_df.shape)
print("val_target:", val_target.shape)
print("test_df:", test_df.shape)
print("test_target:", test_target.shape)


train_df: (4448700, 25)
train_target: (4448700, 1)
val_df: (1317600, 25)
val_target: (1317600, 1)
test_df: (1899361, 25)
test_target: (1899361, 1)


In [ ]:

train_df.head()


CHWC_VLV_DM  ZONE_TEMP_4    SA_CFM  \
fault_type Datetime                                                  
0          2018-01-01 01:00:00          0.0    66.761345 -0.943331   
           2018-01-01 01:01:00          0.0    66.761290 -0.943167   
           2018-01-01 01:02:00          0.0    66.761185 -0.943027   
           2018-01-01 01:03:00          0.0    66.761020 -0.942867   
           2018-01-01 01:04:00          0.0    66.760740 -0.942719   

                                      SF_WAT    MA_TEMP  SYS_CTL  OA_DMPR_DM  \
fault_type Datetime                                                            
0          2018-01-01 01:00:00 -2.791519e-13  66.374680      0.0         0.0   
           2018-01-01 01:01:00 -2.790612e-13  66.374680      0.0         0.0   
           2018-01-01 01:02:00 -2.789696e-13  66.374680      0.0         0.0   
           2018-01-01 01:03:00 -2.788807e-13  66.374626      0.0         0.0   
           2018-01-01 01:04:00 -2.787895e-13  66.374626      0.0         0.0   

                                ZONE_TEMP_2   RA_TEMP      CHWC_VLV  ...  \
fault_type Datetime                                                  ...   
0          2018-01-01 01:00:00    66.767230  68.34175  2.635303e-21  ...   
           2018-01-01 01:01:00    66.766680  68.34786  2.479578e-21  ...   
           2018-01-01 01:02:00    66.766014  68.35378  2.380361e-21  ...   
           2018-01-01 01:03:00    66.765305  68.35948 -1.274259e-21  ...   
           2018-01-01 01:04:00    66.764530  68.36498  2.749987e-21  ...   

                                RA_DMPR_DM    SA_TEMP  RF_CS  RA_DMPR  \
fault_type Datetime                                                     
0          2018-01-01 01:00:00         1.0  66.374680    0.0      1.0   
           2018-01-01 01:01:00         1.0  66.374680    0.0      1.0   
           2018-01-01 01:02:00         1.0  66.374680    0.0      1.0   
           2018-01-01 01:03:00         1.0  66.374626    0.0      1.0   
           2018-01-01 01:04:00         1.0  66.374626    0.0      1.0   

                                  RA_CFM  RF_SPD_DM        RF_WAT  \
fault_type Datetime                                                 
0          2018-01-01 01:00:00  0.853194        0.0 -2.617164e-13   
           2018-01-01 01:01:00  0.853057        0.0 -2.616267e-13   
           2018-01-01 01:02:00  0.852918        0.0 -2.615462e-13   
           2018-01-01 01:03:00  0.852779        0.0 -2.614546e-13   
           2018-01-01 01:04:00  0.852641        0.0 -2.613691e-13   

                                ZONE_TEMP_1  ZONE_TEMP_3  SF_SPD_DM  
fault_type Datetime                                                  
0          2018-01-01 01:00:00     73.94652    67.027100        0.0  
           2018-01-01 01:01:00     73.97777    67.025894        0.0  
           2018-01-01 01:02:00     74.00836    67.024580        0.0  
           2018-01-01 01:03:00     74.03825    67.023150        0.0  
           2018-01-01 01:04:00     74.06741    67.021670        0.0  

[5 rows x 25 columns]

In [ ]:

train_target.head()


target
fault_type Datetime                   
0          2018-01-01 01:00:00       0
           2018-01-01 01:01:00       0
           2018-01-01 01:02:00       0
           2018-01-01 01:03:00       0
           2018-01-01 01:04:00       0

In [ ]:

def ensure_target_series(target: pd.DataFrame | pd.Series | np.ndarray) -> pd.Series:
    """
    Приводит target к Series.
    Поддерживает:
    - Series
    - DataFrame с 1 колонкой
    - ndarray shape (n,) или (n, 1)
    """
    if isinstance(target, pd.Series):
        return target

    if isinstance(target, pd.DataFrame):
        if target.shape[1] != 1:
            raise ValueError(
                f"У target DataFrame {target.shape[1]} столбцов; ожидался 1 столбец."
            )
        return target.iloc[:, 0]

    arr = np.asarray(target)
    if arr.ndim == 2:
        if arr.shape[1] != 1:
            raise ValueError(f"target имеет shape={arr.shape}; ожидался shape (n,) или (n, 1).")
        arr = arr.reshape(-1)
    elif arr.ndim != 1:
        raise ValueError(f"target имеет ndim={arr.ndim}; ожидался 1D или 2D c одним столбцом.")

    return pd.Series(arr)


def prepare_grouped_arrays(df: pd.DataFrame, target) -> tuple[list, list]:
    """
    Возвращает список групп (по fault_type), где каждая группа содержит:
    - X: np.ndarray shape (T, F)
    - y: np.ndarray shape (T,)
    - index: исходный MultiIndex

    Здесь target может быть DataFrame с тем же MultiIndex, что и df.
    """
    x = df.copy().sort_index(level=["fault_type", "Datetime"])
    y = ensure_target_series(target)

    if len(y) != len(x):
        raise ValueError(
            f"Длины df и target не совпадают: len(df)={len(x)}, len(target)={len(y)}"
        )

    # Если target имеет тот же MultiIndex — выравниваем по нему.
    # Иначе считаем, что target уже в том же построчном порядке, что и df.
    if isinstance(y.index, pd.MultiIndex) and y.index.equals(x.index):
        y_sorted = y.loc[x.index]
    else:
        y_sorted = pd.Series(np.asarray(y).reshape(-1), index=x.index)

    num_cols = [c for c in x.columns if pd.api.types.is_numeric_dtype(x[c])]
    feature_names = num_cols

    groups = []
    for fault_value, g in x.groupby(level="fault_type", sort=False):
        g_idx = g.index
        Xg = g[num_cols].to_numpy(dtype=np.float32)
        yg = y_sorted.loc[g_idx].to_numpy()
        yg = np.asarray(yg).reshape(-1)

        groups.append({
            "fault_type": fault_value,
            "X": Xg,
            "y": yg,
            "index": g_idx,
        })

    return groups, feature_names


In [ ]:

train_groups, feature_names = prepare_grouped_arrays(train_df, train_target)
val_groups, _ = prepare_grouped_arrays(val_df, val_target)
test_groups, _ = prepare_grouped_arrays(test_df, test_target)

train_target_s = ensure_target_series(train_target)
if isinstance(train_target_s.index, pd.MultiIndex):
    train_target_s = train_target_s.sort_index(level=["fault_type", "Datetime"])

classes_ = np.sort(pd.unique(train_target_s.to_numpy().reshape(-1)))
n_features = len(feature_names)

print("classes_:", classes_)
print("n_features:", n_features)
print("n_train_groups:", len(train_groups))
print("example X shape:", train_groups[0]["X"].shape)
print("example y shape:", train_groups[0]["y"].shape)
print("example y dtype:", train_groups[0]["y"].dtype)


classes_: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]
n_features: 25
n_train_groups: 15
example X shape: (305220, 25)
example y shape: (305220,)
example y dtype: int64


In [ ]:
def make_window_offsets(window: int, sparse_step: int | None = None) -> np.ndarray:
    """
    Возвращает смещения внутри окна, которые нужно взять.
    - sparse_step=None -> берём все точки окна
    - sparse_step=10, window=100 -> берём позиции 0, 10, ..., 90
    """
    if sparse_step is None:
        return np.arange(window, dtype=np.int64)

    offsets = np.arange(0, window, sparse_step, dtype=np.int64)
    if len(offsets) == 0:
        raise ValueError(
            f"Для window={window} и sparse_step={sparse_step} не получилось построить offsets"
        )
    return offsets


def valid_window_endpoints(group_length: int, window: int) -> np.ndarray:
    if group_length < window:
        return np.array([], dtype=np.int64)
    return np.arange(window - 1, group_length, dtype=np.int64)


def flatten_window(Xg: np.ndarray, end: int, window: int, offsets: np.ndarray) -> np.ndarray:
    start = end - window + 1
    return Xg[start + offsets].reshape(-1)


def build_sampled_train_windows(
    groups,
    window: int,
    total_samples: int,
    random_state: int = 42,
    sparse_step: int | None = None,
    save_selection_path: str | None = None,
):
    """
    Строит raw-window train выборку:
    - окно flatten в вектор длины len(offsets) * n_features
    - сэмпл стратифицирован по классам
    - генерируются только выбранные окна
    - при необходимости сохраняется точный список использованных окон
    """
    rng = np.random.default_rng(random_state)
    offsets = make_window_offsets(window=window, sparse_step=sparse_step)

    class_to_candidates = {}
    for group_idx, g in enumerate(groups):
        endpoints = valid_window_endpoints(len(g["X"]), window)
        if len(endpoints) == 0:
            continue

        labels = np.asarray(g["y"]).reshape(-1)
        for end in endpoints:
            cls = labels[end]
            class_to_candidates.setdefault(cls, []).append((group_idx, int(end)))

    classes = sorted(class_to_candidates.keys())
    if not classes:
        raise ValueError(f"Не найдено ни одного окна для window={window}")

    base_quota = total_samples // len(classes)
    remainder = total_samples % len(classes)

    selected = []
    shortages = 0

    for i, cls in enumerate(classes):
        candidates = class_to_candidates[cls]
        need = base_quota + (1 if i < remainder else 0)
        take = min(need, len(candidates))
        if take > 0:
            idx = rng.choice(len(candidates), size=take, replace=False)
            selected.extend([candidates[j] for j in idx])
        shortages += max(0, need - take)

    if shortages > 0:
        remaining = []
        selected_set = set(selected)
        for cls in classes:
            for item in class_to_candidates[cls]:
                if item not in selected_set:
                    remaining.append(item)

        if remaining:
            take = min(shortages, len(remaining))
            idx = rng.choice(len(remaining), size=take, replace=False)
            selected.extend([remaining[j] for j in idx])

    rng.shuffle(selected)

    n_samples = len(selected)
    if n_samples == 0:
        raise ValueError(f"После сэмплирования не осталось окон для window={window}")

    flat_dim = len(offsets) * groups[0]["X"].shape[1]
    X = np.empty((n_samples, flat_dim), dtype=np.float32)
    y = np.empty(n_samples, dtype=np.int64)
    records = []

    for i, (group_idx, end) in enumerate(selected):
        g = groups[group_idx]
        X[i] = flatten_window(g["X"], end=end, window=window, offsets=offsets)
        y[i] = np.asarray(g["y"][end]).item()

        records.append({
            "sample_id": i,
            "group_idx": group_idx,
            "fault_type": g["fault_type"],
            "window": window,
            "sparse_step": -1 if sparse_step is None else sparse_step,
            "endpoint_pos": int(end),
            "start_pos": int(end - window + 1),
            "label": int(y[i]),
            "endpoint_time": str(g["index"][end][1]),
            "offsets": ','.join(map(str, offsets.tolist())),
        })

    selection_df = pd.DataFrame(records)

    if save_selection_path is not None:
        os.makedirs(os.path.dirname(save_selection_path), exist_ok=True)
        selection_df.to_json(save_selection_path, orient="records", lines=True)
        print(f"Saved train window selection to: {save_selection_path}")

    return X, y, selection_df, offsets


def stream_windows(groups, window: int, batch_size: int = 4096, sparse_step: int | None = None):
    """
    Генератор батчей окон для val/test.
    Ничего не материализует целиком.
    Yield: X_batch, y_batch
    """
    offsets = make_window_offsets(window=window, sparse_step=sparse_step)
    X_parts = []
    y_parts = []

    for g in groups:
        Xg = g["X"]
        yg = np.asarray(g["y"]).reshape(-1)

        if len(Xg) < window:
            continue

        for end in range(window - 1, len(Xg)):
            X_parts.append(flatten_window(Xg, end=end, window=window, offsets=offsets))
            y_parts.append(np.asarray(yg[end]).item())

            if len(X_parts) >= batch_size:
                yield np.asarray(X_parts, dtype=np.float32), np.asarray(y_parts, dtype=np.int64)
                X_parts = []
                y_parts = []

    if X_parts:
        yield np.asarray(X_parts, dtype=np.float32), np.asarray(y_parts, dtype=np.int64)


def evaluate_streaming(
    model,
    groups,
    window: int,
    classes_: np.ndarray,
    batch_size: int = 4096,
    sparse_step: int | None = None,
):
    """
    Streaming evaluation для val/test.
    Поддерживает accuracy, macro_f1, weighted_f1, logloss.
    """
    y_true_all = []
    y_pred_all = []
    y_proba_all = []

    has_proba = hasattr(model, "predict_proba")

    for Xb, yb in stream_windows(groups, window=window, batch_size=batch_size, sparse_step=sparse_step):
        pred = model.predict(Xb)
        y_true_all.append(yb)
        y_pred_all.append(pred)

        if has_proba:
            proba = model.predict_proba(Xb)
            y_proba_all.append(proba)

    y_true = np.concatenate(y_true_all)
    y_pred = np.concatenate(y_pred_all)

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

    if has_proba and len(y_proba_all) > 0:
        y_proba = np.concatenate(y_proba_all)
        metrics["logloss"] = log_loss(y_true, y_proba, labels=classes_)
    else:
        metrics["logloss"] = np.nan

    return metrics, y_true, y_pred


In [ ]:
RANDOM_STATE = 42
TRAIN_SAMPLE_SIZE = 100_000
VAL_BATCH_SIZE = 4_096
TEST_BATCH_SIZE = 4_096
ARTIFACTS_DIR = join("/home/ilya_treyvish/projects/lbnl_fdd/data", "artifacts_raw_windows")


In [ ]:
def run_experiment_for_window(
    window: int,
    train_groups,
    val_groups,
    test_groups,
    train_sample_size: int,
    random_state: int,
    val_batch_size: int = 4096,
    test_batch_size: int = 4096,
    sparse_step: int | None = None,
    version_name: str | None = None,
    artifacts_dir: str | None = None,
):
    if version_name is None:
        version_name = f"raw_window_{window}" if sparse_step is None else f"raw_window_{window}_step{sparse_step}"

    selection_path = None
    if artifacts_dir is not None:
        selection_path = join(artifacts_dir, f"{version_name}_train_window_selection.jsonl")

    X_train, y_train, train_window_selection, offsets = build_sampled_train_windows(
        train_groups,
        window=window,
        total_samples=train_sample_size,
        random_state=random_state,
        sparse_step=sparse_step,
        save_selection_path=selection_path,
    )

    models = [
        ("DummyClassifier", DummyClassifier(strategy="prior")),
        ("ExtraTreesClassifier", ExtraTreesClassifier(
            n_estimators=200,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=random_state,
        )),
        ("HistGradientBoostingClassifier", HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.1,
            random_state=random_state,
        )),
    ]

    rows = []
    fitted_models = {}

    for model_name, model in models:
        print(f"\n=== {version_name} | {model_name} ===")
        t0 = time.perf_counter()
        model.fit(X_train, y_train)
        fit_seconds = time.perf_counter() - t0

        val_metrics, _, _ = evaluate_streaming(
            model,
            val_groups,
            window=window,
            classes_=classes_,
            batch_size=val_batch_size,
            sparse_step=sparse_step,
        )
        test_metrics, _, _ = evaluate_streaming(
            model,
            test_groups,
            window=window,
            classes_=classes_,
            batch_size=test_batch_size,
            sparse_step=sparse_step,
        )

        rows.append({
            "version": version_name,
            "window": window,
            "sparse_step": -1 if sparse_step is None else sparse_step,
            "effective_window_points": len(offsets),
            "model_name": model_name,
            "fit_seconds": fit_seconds,
            "train_samples": len(X_train),
            "train_features": X_train.shape[1],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
            "val_logloss": val_metrics["logloss"],
            "test_accuracy": test_metrics["accuracy"],
            "test_macro_f1": test_metrics["macro_f1"],
            "test_weighted_f1": test_metrics["weighted_f1"],
            "test_logloss": test_metrics["logloss"],
        })
        fitted_models[model_name] = model

    results_df = pd.DataFrame(rows).sort_values(
        ["val_macro_f1", "val_accuracy"], ascending=False
    ).reset_index(drop=True)

    artifacts = {
        "version": version_name,
        "window": window,
        "sparse_step": sparse_step,
        "offsets": offsets,
        "X_train_shape": X_train.shape,
        "y_train_shape": y_train.shape,
        "train_window_selection": train_window_selection,
        "train_window_selection_path": selection_path,
        "results_df": results_df,
        "models": fitted_models,
    }

    del X_train, y_train
    gc.collect()

    return artifacts


In [ ]:
%%time
artifacts_raw_w1 = run_experiment_for_window(
    window=1,
    train_groups=train_groups,
    val_groups=val_groups,
    test_groups=test_groups,
    train_sample_size=TRAIN_SAMPLE_SIZE,
    random_state=RANDOM_STATE,
    val_batch_size=VAL_BATCH_SIZE,
    test_batch_size=TEST_BATCH_SIZE,
    sparse_step=None,
    version_name="raw_window_1",
    artifacts_dir=ARTIFACTS_DIR,
)

artifacts_raw_w1["results_df"]


Saved train window selection to: /home/ilya_treyvish/projects/lbnl_fdd/data/artifacts_raw_windows/raw_window_1_train_window_selection.jsonl

=== raw_window_1 | DummyClassifier ===

=== raw_window_1 | ExtraTreesClassifier ===

=== raw_window_1 | HistGradientBoostingClassifier ===
CPU times: user 17min 27s, sys: 1min 21s, total: 18min 49s
Wall time: 6min 43s


,version,window,sparse_step,effective_window_points,model_name,fit_seconds,train_samples,train_features,val_accuracy,val_macro_f1,val_weighted_f1,val_logloss,test_accuracy,test_macro_f1,test_weighted_f1,test_logloss
0,raw_window_1,1,-1,1,ExtraTreesClassifier,2.021577,50000,25,0.846483,0.845614,0.845614,0.448223,0.780146,0.787443,0.779940,0.602758
1,raw_window_1,1,-1,1,HistGradientBoostingClassifier,8.942326,50000,25,0.839617,0.839765,0.839765,0.421813,0.783813,0.792421,0.783958,0.545815
2,raw_window_1,1,-1,1,DummyClassifier,0.018611,50000,25,0.066667,0.008333,0.008333,2.708050,0.069750,0.008694,0.009096,2.708046


In [ ]:
%%time
artifacts_raw_w10 = run_experiment_for_window(
    window=10,
    train_groups=train_groups,
    val_groups=val_groups,
    test_groups=test_groups,
    train_sample_size=TRAIN_SAMPLE_SIZE,
    random_state=RANDOM_STATE,
    val_batch_size=VAL_BATCH_SIZE,
    test_batch_size=TEST_BATCH_SIZE,
    sparse_step=None,
    version_name="raw_window_10",
    artifacts_dir=ARTIFACTS_DIR,
)

artifacts_raw_w10["results_df"]


Saved train window selection to: /home/ilya_treyvish/projects/lbnl_fdd/data/artifacts_raw_windows/raw_window_10_train_window_selection.jsonl

=== raw_window_10 | DummyClassifier ===

=== raw_window_10 | ExtraTreesClassifier ===

=== raw_window_10 | HistGradientBoostingClassifier ===
CPU times: user 42min 57s, sys: 1min 37s, total: 44min 34s
Wall time: 13min 31s


,version,window,sparse_step,effective_window_points,model_name,fit_seconds,train_samples,train_features,val_accuracy,val_macro_f1,val_weighted_f1,val_logloss,test_accuracy,test_macro_f1,test_weighted_f1,test_logloss
0,raw_window_10,10,-1,10,ExtraTreesClassifier,11.935301,50000,250,0.850026,0.849486,0.849486,0.394850,0.788101,0.794683,0.787384,0.506769
1,raw_window_10,10,-1,10,HistGradientBoostingClassifier,81.428125,50000,250,0.843416,0.843395,0.843395,0.400401,0.785601,0.795642,0.786334,0.520352
2,raw_window_10,10,-1,10,DummyClassifier,0.001218,50000,250,0.066667,0.008333,0.008333,2.708050,0.069750,0.008694,0.009096,2.708046


In [ ]:
%%time
artifacts_raw_w100 = run_experiment_for_window(
    window=100,
    train_groups=train_groups,
    val_groups=val_groups,
    test_groups=test_groups,
    train_sample_size=TRAIN_SAMPLE_SIZE,
    random_state=RANDOM_STATE,
    val_batch_size=VAL_BATCH_SIZE,
    test_batch_size=TEST_BATCH_SIZE,
    sparse_step=None,
    version_name="raw_window_100",
    artifacts_dir=ARTIFACTS_DIR,
)

artifacts_raw_w100["results_df"]


Saved train window selection to: /home/ilya_treyvish/projects/lbnl_fdd/data/artifacts_raw_windows/raw_window_100_train_window_selection.jsonl

=== raw_window_100 | DummyClassifier ===

=== raw_window_100 | ExtraTreesClassifier ===

=== raw_window_100 | HistGradientBoostingClassifier ===
CPU times: user 1h 59min 37s, sys: 3min 16s, total: 2h 2min 54s
Wall time: 33min 4s


,version,window,sparse_step,effective_window_points,model_name,fit_seconds,train_samples,train_features,val_accuracy,val_macro_f1,val_weighted_f1,val_logloss,test_accuracy,test_macro_f1,test_weighted_f1,test_logloss
0,raw_window_100,100,-1,100,HistGradientBoostingClassifier,659.230621,50000,2500,0.863480,0.863574,0.863574,0.352309,0.812587,0.821828,0.813582,0.453709
1,raw_window_100,100,-1,100,ExtraTreesClassifier,44.136363,50000,2500,0.863170,0.862651,0.862651,0.406996,0.809150,0.816051,0.808569,0.477990
2,raw_window_100,100,-1,100,DummyClassifier,0.001266,50000,2500,0.066667,0.008333,0.008333,2.708050,0.069752,0.008694,0.009096,2.708046


In [ ]:
%%time
artifacts_raw_w100_sparse = run_experiment_for_window(
    window=100,
    train_groups=train_groups,
    val_groups=val_groups,
    test_groups=test_groups,
    train_sample_size=TRAIN_SAMPLE_SIZE,
    random_state=RANDOM_STATE,
    val_batch_size=VAL_BATCH_SIZE,
    test_batch_size=TEST_BATCH_SIZE,
    sparse_step=10,
    version_name="raw_window_100_sparse_step10",
    artifacts_dir=ARTIFACTS_DIR,
)

artifacts_raw_w100_sparse["results_df"]


Saved train window selection to: /home/ilya_treyvish/projects/lbnl_fdd/data/artifacts_raw_windows/raw_window_100_sparse_step10_train_window_selection.jsonl

=== raw_window_100_sparse_step10 | DummyClassifier ===

=== raw_window_100_sparse_step10 | ExtraTreesClassifier ===

=== raw_window_100_sparse_step10 | HistGradientBoostingClassifier ===
CPU times: user 27min 6s, sys: 1min 17s, total: 28min 24s
Wall time: 8min 53s


,version,window,sparse_step,effective_window_points,model_name,fit_seconds,train_samples,train_features,val_accuracy,val_macro_f1,val_weighted_f1,val_logloss,test_accuracy,test_macro_f1,test_weighted_f1,test_logloss
0,raw_window_100_sparse_step10,100,10,10,HistGradientBoostingClassifier,59.178224,50000,250,0.862401,0.862614,0.862614,0.361282,0.809392,0.818886,0.810567,0.475031
1,raw_window_100_sparse_step10,100,10,10,ExtraTreesClassifier,10.310575,50000,250,0.859335,0.858801,0.858801,0.378840,0.803024,0.809935,0.802255,0.500158
2,raw_window_100_sparse_step10,100,10,10,DummyClassifier,0.002522,50000,250,0.066667,0.008333,0.008333,2.708050,0.069752,0.008694,0.009096,2.708046


In [ ]:
comparison = pd.concat(
    [
        artifacts_raw_w1["results_df"],
        artifacts_raw_w10["results_df"],
        artifacts_raw_w100["results_df"],
        artifacts_raw_w100_sparse["results_df"],
    ],
    axis=0,
)

comparison = comparison[
    [
        "version",
        "window",
        "sparse_step",
        "effective_window_points",
        "model_name",
        "fit_seconds",
        "train_samples",
        "train_features",
        "val_accuracy",
        "val_macro_f1",
        "val_weighted_f1",
        "val_logloss",
        "test_accuracy",
        "test_macro_f1",
        "test_weighted_f1",
        "test_logloss",
    ]
].sort_values(["val_macro_f1", "val_accuracy"], ascending=False).reset_index(drop=True)

comparison


,version,window,sparse_step,effective_window_points,model_name,fit_seconds,train_samples,train_features,val_accuracy,val_macro_f1,val_weighted_f1,val_logloss,test_accuracy,test_macro_f1,test_weighted_f1,test_logloss
0,raw_window_100,100,-1,100,HistGradientBoostingClassifier,659.230621,50000,2500,0.863480,0.863574,0.863574,0.352309,0.812587,0.821828,0.813582,0.453709
1,raw_window_100,100,-1,100,ExtraTreesClassifier,44.136363,50000,2500,0.863170,0.862651,0.862651,0.406996,0.809150,0.816051,0.808569,0.477990
2,raw_window_100_sparse_step10,100,10,10,HistGradientBoostingClassifier,59.178224,50000,250,0.862401,0.862614,0.862614,0.361282,0.809392,0.818886,0.810567,0.475031
3,raw_window_100_sparse_step10,100,10,10,ExtraTreesClassifier,10.310575,50000,250,0.859335,0.858801,0.858801,0.378840,0.803024,0.809935,0.802255,0.500158
4,raw_window_10,10,-1,10,ExtraTreesClassifier,11.935301,50000,250,0.850026,0.849486,0.849486,0.394850,0.788101,0.794683,0.787384,0.506769
5,raw_window_1,1,-1,1,ExtraTreesClassifier,2.021577,50000,25,0.846483,0.845614,0.845614,0.448223,0.780146,0.787443,0.779940,0.602758
6,raw_window_10,10,-1,10,HistGradientBoostingClassifier,81.428125,50000,250,0.843416,0.843395,0.843395,0.400401,0.785601,0.795642,0.786334,0.520352
7,raw_window_1,1,-1,1,HistGradientBoostingClassifier,8.942326,50000,25,0.839617,0.839765,0.839765,0.421813,0.783813,0.792421,0.783958,0.545815
8,raw_window_1,1,-1,1,DummyClassifier,0.018611,50000,25,0.066667,0.008333,0.008333,2.708050,0.069750,0.008694,0.009096,2.708046
9,raw_window_10,10,-1,10,DummyClassifier,0.001218,50000,250,0.066667,0.008333,0.008333,2.708050,0.069750,0.008694,0.009096,2.708046


## Необязательная ячейка: detailed report для лучшей модели по каждой конфигурации


In [ ]:
experiment_specs = [
    ("raw_window_1", artifacts_raw_w1, 1, None),
    ("raw_window_10", artifacts_raw_w10, 10, None),
    ("raw_window_100", artifacts_raw_w100, 100, None),
    ("raw_window_100_sparse_step10", artifacts_raw_w100_sparse, 100, 10),
]

for version_name, artifacts, window, sparse_step in experiment_specs:
    best_name = artifacts["results_df"].iloc[0]["model_name"]
    best_model = artifacts["models"][best_name]

    _, y_true_val, y_pred_val = evaluate_streaming(
        best_model,
        val_groups,
        window=window,
        classes_=classes_,
        batch_size=VAL_BATCH_SIZE,
        sparse_step=sparse_step,
    )

    print("=" * 100)
    print(f"Best model for {version_name}: {best_name}")
    print(classification_report(y_true_val, y_pred_val, zero_division=0))


Best model for raw_window_1: ExtraTreesClassifier
              precision    recall  f1-score   support

           0       0.53      0.54      0.54     87840
           1       0.85      0.78      0.81     87840
           2       0.89      0.96      0.93     87840
           3       0.75      0.71      0.73     87840
           4       0.74      0.84      0.79     87840
           5       0.71      0.62      0.66     87840
           6       0.98      0.98      0.98     87840
           7       1.00      0.99      0.99     87840
           8       1.00      1.00      1.00     87840
           9       1.00      1.00      1.00     87840
          10       0.67      0.72      0.69     87840
          11       1.00      1.00      1.00     87840
          12       1.00      1.00      1.00     87840
          13       0.99      0.99      0.99     87840
          14       0.60      0.57      0.58     87840

    accuracy                           0.85   1317600
   macro avg       0.85      0